# Advanced: Multi-Strategy Text Semantic Categorization with PAMOLA.CORE

**Goal**: Master all 3 text categorization strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Dictionary-based categorization with custom hierarchies
- **Strategy 2**: NER-based entity extraction for unmatched texts
- **Strategy 3**: Clustering-based semantic grouping
- Calculate advanced categorization metrics
- Analyze match effectiveness and coverage
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of NLP concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 02_text_semantic_categorizer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.text import (
        TextSemanticCategorizerOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 5 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'text_semantic_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Job titles with variety
    job_titles = [
        'Senior Software Engineer', 'Software Engineer', 'Junior Software Engineer',
        'Senior Data Scientist', 'Data Scientist', 'Machine Learning Engineer',
        'DevOps Engineer', 'Site Reliability Engineer', 'Cloud Architect',
        'Full Stack Developer', 'Backend Developer', 'Frontend Developer',
        'Mobile Developer', 'iOS Developer', 'Android Developer',
        'Senior Data Engineer', 'Data Engineer', 'Data Analyst',
        'Business Intelligence Analyst', 'Analytics Engineer',
        'Product Manager', 'Senior Product Manager', 'Technical Program Manager',
        'Engineering Manager', 'Director of Engineering', 'VP of Engineering',
        'UX Designer', 'UI Designer', 'Product Designer', 'UX Researcher',
        'QA Engineer', 'Security Engineer', 'Database Administrator',
        'System Administrator', 'Network Engineer', 'Technical Writer'
    ]
    
    companies = [
        'Tech Corp Inc', 'Data Solutions LLC', 'Cloud Systems Group',
        'Innovation Labs', 'Digital Services Co', 'Analytics Partners',
        'Software Factory', 'AI Research Institute', 'Platform Systems',
        'Enterprise Solutions', 'Mobile Apps Ltd', 'Security First Inc'
    ]
    
    skills = [
        'Python', 'Java', 'JavaScript', 'C++', 'Go', 'Rust',
        'React', 'Angular', 'Vue.js', 'Node.js', 'Django', 'Flask',
        'TensorFlow', 'PyTorch', 'Scikit-learn', 'Pandas', 'NumPy',
        'AWS', 'Azure', 'Google Cloud', 'Docker', 'Kubernetes',
        'SQL', 'MongoDB', 'PostgreSQL', 'Redis', 'Elasticsearch'
    ]
    
    df = pd.DataFrame({
        'resume_id': range(1, n_records + 1),
        'job_title_current': np.random.choice(job_titles, n_records),
        'company_name': np.random.choice(companies, n_records),
        'primary_skill': np.random.choice(skills, n_records),
        'experience_years': np.random.randint(0, 20, n_records)
    })
    
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Custom Dictionary

**How to use:**
- Run to check/create job roles dictionary
- Uses existing file if found
- Creates example if missing

**What you'll see:**
- File status (✓ found or 🔧 created)
- File location path
- Structure summary (categories, aliases)
- Usage instructions

**Note:** Edit file to match your data before processing

In [ ]:
examples_dir = project_root / 'examples'
dictionary_path = examples_dir / 'data_examples' / 'job_roles_dictionary.json'

if dictionary_path.exists():
    print(f"✓ Found existing dictionary: {dictionary_path}")
    print(f"📂 Using existing file for Strategy 1\n")
    
    with open(dictionary_path, 'r') as f:
        existing_dict = json.load(f)
    
    print("📊 Existing Dictionary Info:")
    print(f"  Format:   v{existing_dict.get('format_version', 'N/A')}")
    print(f"  Type:     {existing_dict.get('entity_type', 'N/A')}")
    print(f"  Categories: {len(existing_dict.get('categories', {}))}")
    
    print("\n💡 To use different dictionary, replace or rename it.")
    print("=" * 80)

else:
    print("⚠️  Dictionary not found!")
    print("🔧 Creating example job roles dictionary...\n")
    print("=" * 80)

    job_dictionary = {
        "format_version": "1.0",
        "entity_type": "job",
        "description": "Hierarchical categorization for job roles",
        "categories": {
            "Software_Engineer": {
                "domain": "Engineering",
                "seniority": ["Junior", "Mid", "Senior"],
                "aliases": [
                    "software engineer", "software developer", "programmer",
                    "application developer", "systems engineer"
                ]
            },
            "Data_Scientist": {
                "domain": "Data Science",
                "seniority": ["Junior", "Mid", "Senior"],
                "aliases": [
                    "data scientist", "ml engineer", "machine learning engineer",
                    "ai engineer", "research scientist"
                ]
            },
            "Data_Engineer": {
                "domain": "Data Engineering",
                "seniority": ["Junior", "Mid", "Senior"],
                "aliases": [
                    "data engineer", "etl engineer", "data pipeline engineer",
                    "analytics engineer"
                ]
            },
            "Frontend_Developer": {
                "domain": "Engineering",
                "seniority": ["Junior", "Mid", "Senior"],
                "aliases": [
                    "frontend developer", "front-end developer", "ui developer",
                    "web developer", "javascript developer"
                ]
            },
            "Product_Manager": {
                "domain": "Product",
                "seniority": ["Junior", "Mid", "Senior"],
                "aliases": [
                    "product manager", "pm", "technical product manager",
                    "product owner"
                ]
            }
        }
    }
    
    try:
        dictionary_path.parent.mkdir(parents=True, exist_ok=True)
        with open(dictionary_path, 'w') as f:
            json.dump(job_dictionary, f, indent=2)
        print(f"✓ Example dictionary created: {dictionary_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {dictionary_path} (file may be open)")

    print(f"\n📊 Statistics:")
    print(f"  Categories: {len(job_dictionary['categories'])}")
    total_aliases = sum(len(cat['aliases']) for cat in job_dictionary['categories'].values())
    print(f"  Total aliases: {total_aliases}")
    print("\n💡 Edit this file to match your job categories!")
    print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names
   - `strategy1_field`: For dictionary-based
   - `strategy2_field`: For NER-based
   - `strategy3_field`: For clustering-based
2. Run to validate fields and setup

**What you'll see:**
- Field validation (✓ marks)
- Unique value counts per field
- Task directory created (✓)
- Environment ready (✓)

**Note:** All fields must exist in dataset

In [ ]:
FIELD_CONFIG = {
    "strategy1_field": "job_title_current",
    "strategy2_field": "company_name",
    "strategy3_field": "primary_skill"
}

print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available: {', '.join(df.columns)}"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique)")

def create_config_kwargs():
    return {"dataset_name": "main_dataset"}

task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy text categorization",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

print(f"\n✅ Shared environment ready!")

## STRATEGY 1: Dictionary-Based

**How to use:**
- Uses custom dictionary from Step 3
- Matches text against predefined categories
- Run to execute dictionary-based strategy

**Key parameters:**
- `entity_type='job'`: Uses job roles dictionary
- `use_ner=False`: Dictionary only
- `perform_clustering=False`: No clustering

**What you'll see:**
- Progress bar (6 steps)
- Completion time (1-3 seconds)
- Match statistics

**Note:** Fast but limited to dictionary coverage

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: DICTIONARY-BASED")
print("=" * 80 + "\n")

tracker_s1 = HierarchicalProgressTracker(
    total=6, description="Strategy 1: Dictionary", unit="steps"
)

operation_s1 = TextSemanticCategorizerOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    id_field="resume_id",
    entity_type='job',
    dictionary_path=str(dictionary_path),
    perform_categorization=True,
    use_ner=False,
    perform_clustering=False,
    use_vectorization=True,
    parallel_processes=2,
    generate_visualization=True,
    save_output=True
)

print("✓ Strategy 1 configured\n")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_dictionary',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

if result_s1.metrics:
    print(f"\n📊 Metrics:")
    print(f"   Total:      {result_s1.metrics.get('total_records', 0):,}")
    print(f"   Matched:    {result_s1.metrics.get('num_matched', 0):,}")
    print(f"   Match rate: {result_s1.metrics.get('percentage_matched', 0):.1f}%")

## STRATEGY 2: NER-Based

**How to use:**
- Combines dictionary + NER
- NER extracts entities from unmatched
- Run to execute NER-enhanced strategy

**Key parameters:**
- `entity_type='organization'`: For companies
- `use_ner=True`: Enable NER
- `perform_clustering=False`: No clustering

**What you'll see:**
- Progress bar (6 steps)
- Completion time (2-4 seconds)
- Match breakdown (dict + NER)

**Note:** Better coverage than dictionary alone

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: NER-BASED")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6, description="Strategy 2: NER", unit="steps"
)

operation_s2 = TextSemanticCategorizerOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    id_field="resume_id",
    entity_type='organization',
    dictionary_path=None,
    perform_categorization=True,
    use_ner=True,
    perform_clustering=False,
    use_vectorization=True,
    parallel_processes=2,
    generate_visualization=True,
    save_output=True
)

print("✓ Strategy 2 configured\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_ner',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

if result_s2.metrics:
    print(f"\n📊 Metrics:")
    print(f"   Total:      {result_s2.metrics.get('total_records', 0):,}")
    print(f"   Dict:       {result_s2.metrics.get('num_matched', 0):,}")
    print(f"   NER:        {result_s2.metrics.get('num_ner_matched', 0):,}")
    print(f"   Match rate: {result_s2.metrics.get('percentage_matched', 0):.1f}%")

## STRATEGY 3: Clustering-Based

**How to use:**
- Uses all three methods: dict + NER + clustering
- Clusters remaining by similarity
- Run to execute comprehensive strategy

**Key parameters:**
- `entity_type='skill'`: For technical skills
- `use_ner=True`: Enable NER
- `perform_clustering=True`: Enable clustering
- `clustering_threshold=0.7`: Similarity (0-1)

**What you'll see:**
- Progress bar (6 steps)
- Completion time (3-5 seconds)
- Full breakdown (dict + NER + cluster)

**Note:** Maximum coverage, highest quality

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: CLUSTERING-BASED")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6, description="Strategy 3: Clustering", unit="steps"
)

operation_s3 = TextSemanticCategorizerOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    id_field="resume_id",
    entity_type='skill',
    dictionary_path=None,
    perform_categorization=True,
    use_ner=True,
    perform_clustering=True,
    clustering_threshold=0.7,
    use_vectorization=True,
    parallel_processes=2,
    generate_visualization=True,
    save_output=True
)

print("✓ Strategy 3 configured\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_clustering',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

if result_s3.metrics:
    print(f"\n📊 Metrics:")
    print(f"   Total:      {result_s3.metrics.get('total_records', 0):,}")
    print(f"   Dict:       {result_s3.metrics.get('num_matched', 0):,}")
    print(f"   NER:        {result_s3.metrics.get('num_ner_matched', 0):,}")
    print(f"   Clustered:  {result_s3.metrics.get('num_auto_clustered', 0):,}")
    print(f"   Match rate: {result_s3.metrics.get('percentage_matched', 0):.1f}%")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Match coverage comparison
- Coverage improvement metrics

**Note:** Strategy 3 has highest coverage but longest time

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

if result_s1.metrics and result_s2.metrics and result_s3.metrics:
    s1_pct = result_s1.metrics.get('percentage_matched', 0)
    s2_pct = result_s2.metrics.get('percentage_matched', 0)
    s3_pct = result_s3.metrics.get('percentage_matched', 0)
    
    print(f"\n📈 Match Coverage:")
    print(f"  Strategy 1: {s1_pct:5.1f}% - Dictionary only")
    print(f"  Strategy 2: {s2_pct:5.1f}% - Dictionary + NER")
    print(f"  Strategy 3: {s3_pct:5.1f}% - All methods")
    
    print(f"\n💡 S3 is {s3_pct - s1_pct:.1f}% better than S1")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for artifact inspection
- Focus on validation points

**What you'll see (per strategy):**
1. **Metrics**: Match rates, method breakdown
2. **Visualizations**: Category distributions

**Note:** Only newest files shown

In [ ]:
strategy_dirs = [
    ('strategy1_dictionary', 'Strategy 1: Dictionary'),
    ('strategy2_ner', 'Strategy 2: NER'),
    ('strategy3_clustering', 'Strategy 3: Clustering')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]
            target_files = filtered if filtered else metrics_files
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            
            print(f"\n📄 METRICS: {target_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(target_files[0], 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                print(f"   Total records        : {metrics.get('total_records', 0):,}")
                print(f"   Dictionary matched   : {metrics.get('num_matched', 0):,}")
                print(f"   NER matched          : {metrics.get('num_ner_matched', 0):,}")
                print(f"   Auto-clustered       : {metrics.get('num_auto_clustered', 0):,}")
                print(f"   Unresolved           : {metrics.get('num_unresolved', 0):,}")
                print(f"   Match rate           : {metrics.get('percentage_matched', 0):.2f}%")
                print(f"   Avg text length      : {metrics.get('avg_text_length', 0):.2f}")
                print(f"   Predominant language : {metrics.get('predominant_language', 'N/A')}")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Text Semantic Analysis (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir and output_dir.exists():
        analysis_files = sorted(
            list(output_dir.glob('*_text_semantic_analysis_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if analysis_files:
            print(f"\n📊 SEMANTIC ANALYSIS: {analysis_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(analysis_files[0], 'r') as f:
                    analysis = json.load(f)
                
                # Field info
                print(f"   Field analyzed       : {analysis.get('field_name', 'N/A')}")
                print(f"   Entity type          : {analysis.get('entity_type', 'N/A')}")
                
                # Null/Empty analysis
                if 'null_empty_analysis' in analysis:
                    null_analysis = analysis['null_empty_analysis']
                    print(f"\n   📋 Data Quality:")
                    print(f"      Total records     : {null_analysis.get('total_records', 0):,}")
                    
                    null_vals = null_analysis.get('null_values', {})
                    empty_vals = null_analysis.get('empty_strings', {})
                    whitespace_vals = null_analysis.get('whitespace_strings', {})
                    actual_data = null_analysis.get('actual_data', {})
                    
                    print(f"      Null values       : {null_vals.get('count', 0):,} ({null_vals.get('percentage', 0):.2f}%)")
                    print(f"      Empty strings     : {empty_vals.get('count', 0):,} ({empty_vals.get('percentage', 0):.2f}%)")
                    print(f"      Whitespace only   : {whitespace_vals.get('count', 0):,} ({whitespace_vals.get('percentage', 0):.2f}%)")
                    print(f"      Actual data       : {actual_data.get('count', 0):,} ({actual_data.get('percentage', 0):.2f}%)")
                
                # Length statistics
                if 'length_stats' in analysis:
                    length_stats = analysis['length_stats']
                    print(f"\n   📏 Length Statistics:")
                    print(f"      Min length        : {length_stats.get('min', 0)}")
                    print(f"      Max length        : {length_stats.get('max', 0)}")
                    print(f"      Mean length       : {length_stats.get('mean', 0):.2f}")
                    print(f"      Median length     : {length_stats.get('median', 0):.2f}")
                    print(f"      Std deviation     : {length_stats.get('std', 0):.2f}")
                    
                    # Length distribution
                    if 'length_distribution' in length_stats:
                        print(f"\n      Length Distribution:")
                        for range_key, percentage in length_stats['length_distribution'].items():
                            if percentage > 0:
                                print(f"         {range_key:10s}   : {percentage:.2f}%")
                
                # Language analysis
                if 'language_analysis' in analysis:
                    lang_analysis = analysis['language_analysis']
                    print(f"\n   🌍 Language Analysis:")
                    print(f"      Predominant       : {lang_analysis.get('predominant_language', 'N/A')}")
                    
                    if 'language_distribution' in lang_analysis:
                        lang_dist = lang_analysis['language_distribution']
                        print(f"      Distribution:")
                        for lang, percentage in sorted(lang_dist.items(), key=lambda x: x[1], reverse=True):
                            print(f"         {lang:5s}         : {percentage*100:.2f}%")
                
                # Match summary
                if 'match_summary' in analysis:
                    match_summary = analysis['match_summary']
                    print(f"\n   🎯 Matching Summary:")
                    print(f"      Total texts       : {match_summary.get('total_texts', 0):,}")
                    print(f"      Dictionary matched: {match_summary.get('num_matched', 0):,}")
                    print(f"      NER matched       : {match_summary.get('num_ner_matched', 0):,}")
                    print(f"      Auto-clustered    : {match_summary.get('num_auto_clustered', 0):,}")
                    print(f"      Unresolved        : {match_summary.get('num_unresolved', 0):,}")
                    print(f"      Match percentage  : {match_summary.get('percentage_matched', 0):.2f}%")
                    
                    # Top categories
                    if 'top_categories' in match_summary and match_summary['top_categories']:
                        print(f"\n      Top Categories:")
                        for idx, cat in enumerate(match_summary['top_categories'][:5], 1):
                            print(f"         {idx}. {cat}")
                    
                    # Top aliases
                    if 'top_aliases' in match_summary and match_summary['top_aliases']:
                        print(f"\n      Top Aliases:")
                        for idx, alias in enumerate(match_summary['top_aliases'][:5], 1):
                            print(f"         {idx}. {alias}")
                
                # Category distribution
                if 'category_distribution' in analysis and analysis['category_distribution']:
                    cat_dist = analysis['category_distribution']
                    print(f"\n   📊 Category Distribution ({len(cat_dist)} categories):")
                    
                    sorted_cats = sorted(cat_dist.items(), key=lambda x: x[1], reverse=True)
                    for cat, count in sorted_cats[:10]:
                        print(f"      • {cat:40s}: {count:,}")
                    
                    if len(cat_dist) > 10:
                        print(f"      ... and {len(cat_dist) - 10} more categories")
                
                # Unresolved terms sample
                if 'unresolved_terms' in analysis and analysis['unresolved_terms']:
                    unresolved = analysis['unresolved_terms']
                    print(f"\n   ❓ Unresolved Terms Sample (showing {min(10, len(unresolved))} of {len(unresolved)}):")
                    
                    for idx, term in enumerate(unresolved[:10], 1):
                        record_id = term.get('record_id', 'N/A')
                        text = term.get('text', 'N/A')
                        normalized = term.get('normalized_text', 'N/A')
                        lang = term.get('language', 'N/A')
                        print(f"      {idx:2d}. [{record_id}] {text}")
                        if normalized != text.lower():
                            print(f"          Normalized: {normalized}")
                
            except Exception as e:
                print(f"   ⚠️  Error reading semantic analysis: {e}")
                import traceback
                traceback.print_exc()
    
    # 3. Dictionaries Overview (from dictionaries/)
    dict_dir = strategy_dir / 'dictionaries'
    if dict_dir and dict_dir.exists():
        dict_files = sorted(
            [f for f in dict_dir.iterdir() if f.is_file()],
            key=lambda f: f.stat().st_mtime,
            reverse=True
        )
        
        if dict_files:
            print(f"\n📚 DICTIONARIES: {len(dict_files)} files")
            print("   " + "-" * 60)
            
            for df_file in dict_files[:10]:  # Show first 10
                try:
                    # Count rows efficiently
                    with open(df_file, 'r') as f:
                        row_count = sum(1 for _ in f) - 1  # Minus header
                    
                    # Get file size
                    file_size = df_file.stat().st_size / 1024  # KB
                    
                    print(f"   • {df_file.name:50s}")
                    print(f"      Rows: {row_count:,} | Size: {file_size:.2f} KB")
                    
                except Exception as e:
                    print(f"   • {df_file.name}")
                    print(f"      Error: {e}")
            
            if len(dict_files) > 10:
                print(f"\n   ... and {len(dict_files) - 10} more dictionaries")
    
    # 4. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Coverage Metrics

**How to use:**
- Calculate categorization coverage
- Compare effectiveness across methods
- Run AFTER all strategies complete

**What you'll see:**
- Match method breakdown per strategy
- Coverage progression (S1 → S2 → S3)
- Improvement ratios

**Note:** Higher coverage = better completeness

In [ ]:
print("\n" + "=" * 80)
print("📊 COVERAGE METRICS")
print("=" * 80 + "\n")

try:
    if result_s1.metrics and result_s2.metrics and result_s3.metrics:
        print("📈 STRATEGY 1: Dictionary-Based")
        print(f"   Dictionary: {result_s1.metrics.get('num_matched', 0):4d}")
        print(f"   Coverage:   {result_s1.metrics.get('percentage_matched', 0):.1f}%")
        
        print(f"\n📈 STRATEGY 2: NER-Based")
        print(f"   Dictionary: {result_s2.metrics.get('num_matched', 0):4d}")
        print(f"   NER:        {result_s2.metrics.get('num_ner_matched', 0):4d}")
        print(f"   Coverage:   {result_s2.metrics.get('percentage_matched', 0):.1f}%")
        
        print(f"\n📈 STRATEGY 3: Clustering-Based")
        print(f"   Dictionary: {result_s3.metrics.get('num_matched', 0):4d}")
        print(f"   NER:        {result_s3.metrics.get('num_ner_matched', 0):4d}")
        print(f"   Clustered:  {result_s3.metrics.get('num_auto_clustered', 0):4d}")
        print(f"   Coverage:   {result_s3.metrics.get('percentage_matched', 0):.1f}%")
        
        s1_pct = result_s1.metrics.get('percentage_matched', 0)
        s2_pct = result_s2.metrics.get('percentage_matched', 0)
        s3_pct = result_s3.metrics.get('percentage_matched', 0)
        
        print(f"\n🎯 IMPROVEMENT:")
        print(f"   S2 vs S1: +{s2_pct - s1_pct:.1f}% (NER benefit)")
        print(f"   S3 vs S1: +{s3_pct - s1_pct:.1f}% (Full benefit)")
except NameError:
    print("⚠️  Run all strategies first!")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports categorized datasets

**What you'll see (per strategy):**
- Export confirmation
- Preview (first 5 rows)
- Match method distribution

**Note:** Files in dictionaries/ subfolders

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL DATA")
print("=" * 80)

for dir_name, title in strategy_dirs:
    dict_dir = task_dir / dir_name / 'dictionaries'
    if not dict_dir.exists():
        continue
    
    print(f"\n{title}:")
    
    mapping_files = sorted(
        list(dict_dir.glob('*_category_mappings_*.csv')),
        key=lambda x: x.stat().st_mtime, reverse=True
    )
    
    if mapping_files:
        mapping_file = mapping_files[0]
        print(f"✅ {mapping_file.name}")
        
        try:
            mapping_df = pd.read_csv(mapping_file)
            print(f"   Rows: {len(mapping_df):,}")
            print(f"\n📊 Preview:")
            print(mapping_df.head())
            
            if 'match_method' in mapping_df.columns:
                print(f"\n📈 Match Methods:")
                for method, count in mapping_df['match_method'].value_counts().items():
                    print(f"   {method:15s}: {count:4d}")
        except Exception as e:
            print(f"   ⚠️  Error: {e}")

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 categorization strategies implemented
- ✅ Coverage metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production artifacts generated

**Key takeaways:**
- Dictionary: Fast, high precision, limited coverage
- NER: Broader coverage, entity extraction
- Clustering: Maximum coverage, semantic grouping

**Strategy recommendations:**
- **Use Strategy 1** when: Quality dictionary available, speed critical
- **Use Strategy 2** when: Dictionary incomplete, need entity extraction
- **Use Strategy 3** when: Maximum coverage needed, semantic patterns important

**Next steps:**
- Test with your datasets
- Tune clustering thresholds
- Build custom dictionaries
- Deploy to production

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)